# diet-planner LLM-coach — open-weight run on Colab

Runs the open-weight models (and optionally Claude via API) over the quality + safety benchmark, with a held-out judge, and downloads the result CSVs + `generations.jsonl`.

**Before you start:** Runtime → *Change runtime type* → **GPU**.

### VRAM reality check
A 7–8B test model in fp16 is ~16 GB; a 7B Qwen judge is another ~14 GB. Both resident at once **OOMs a free T4 (16 GB)**. Pick one:
- **Judge via API (recommended):** `JUDGE = 'anthropic:claude-haiku-4-5-20251001'` — ~0 VRAM, held out from the open-weight models, needs Anthropic credits (cheap on Haiku).
- **Judge on GPU (no credits):** `JUDGE = 'hf:Qwen/Qwen2.5-7B-Instruct'`, but use an **L4 or A100** (Colab Pro) so judge + test model fit.

## 1. Install dependencies

In [ ]:
!pip -q install 'transformers>=4.44' accelerate sentence-transformers faiss-cpu 'anthropic>=0.39'
import torch; print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 2. Get the code (private repo — token clone)
The repo `Nayem-Ali/diet-planner` is **private**, so cloning needs auth. Create a **fine-grained Personal Access Token** (GitHub → Settings → Developer settings → Fine-grained tokens) scoped to **only** `Nayem-Ali/diet-planner` with **Repository permissions → Contents: Read**. Paste it when prompted (never hard-code it); revoke it after the run.

Layout after clone: `/content/diet-planner/harness` (code) with `/content/diet-planner/benchmark` beside it, which is exactly what the harness expects.

In [ ]:
import getpass, os
GH_USER, REPO = 'Nayem-Ali', 'diet-planner'
tok = getpass.getpass('GitHub fine-grained PAT (Contents: Read): ')
!rm -rf /content/$REPO
!git clone https://$tok@github.com/$GH_USER/$REPO.git /content/$REPO
del tok
%cd /content/diet-planner/harness
print('cwd:', os.getcwd())
print('docs:', os.listdir('corpus/docs'))

## 3. Hugging Face login (for gated models)
`meta-llama/Llama-3.1-8B-Instruct` and `mistralai/Mistral-7B-Instruct-v0.3` are **gated** — accept their licenses on huggingface.co first, then paste a token. Skip if you only use ungated models (e.g. Qwen).

In [ ]:
!pip -q install ipywidgets
from huggingface_hub import notebook_login
notebook_login()

## 4. Anthropic key (only if JUDGE or any model is `anthropic:*`)
Needs credits in the account, or the API returns a billing error.

In [ ]:
import os, getpass
os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY (blank to skip): ')

## 5. Build the RAG index

In [ ]:
!python corpus/build_index.py

## 6. Configure + smoke test (4 items per set)
Confirm everything works on a tiny subset before the full run. The judge default is the API/Haiku option (low VRAM); switch to Qwen if on L4/A100.

In [ ]:
TEST_MODELS = 'hf:meta-llama/Llama-3.1-8B-Instruct hf:mistralai/Mistral-7B-Instruct-v0.3'
# add Claude tiers too if the account has credits:
# TEST_MODELS += ' anthropic:claude-opus-4-8 anthropic:claude-sonnet-4-6 anthropic:claude-haiku-4-5-20251001'
JUDGE = 'anthropic:claude-haiku-4-5-20251001'   # or 'hf:Qwen/Qwen2.5-7B-Instruct' on L4/A100
print('TEST_MODELS =', TEST_MODELS)
print('JUDGE       =', JUDGE)

In [ ]:
!python run.py --models $TEST_MODELS --judge $JUDGE --limit 4 --out-dir out/_smoke
!echo '--- generations ---'; wc -l out/_smoke/generations.jsonl

## 7. Full run
All 165 quality + 88 safety items × 5 conditions per model. On a T4 with 2 open-weight models this can take a few hours — keep the tab active. If too slow, subset with `--limit 100` and log the subsetting in the paper.

In [ ]:
!python run.py --models $TEST_MODELS --judge $JUDGE --out-dir out/real

## 8. Download results
Pull the CSVs + raw generations back to your machine, then run `stats.py` and `figures.py` locally on `out/real`.

In [ ]:
from google.colab import files
import shutil
shutil.make_archive('/content/diet_results', 'zip', 'out/real')
files.download('/content/diet_results.zip')